In [1]:
import tensorflow as tf, numpy as np, time
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")  # good on Apple Silicon

B, T, Vsrc, Vtgt = 512, 50, 500, 500   # from your config
H, E = 64, 128

# synthetic data that matches your shapes
X = np.random.randint(0, Vsrc, size=(B, T)).astype('int32')
dec_in = np.random.randint(0, Vtgt, size=(B, T)).astype('int32')
dec_out = np.random.randint(0, Vtgt, size=(B, T)).astype('int32')

# build your model exactly once (reuse your build_model if you want)
enc_in = tf.keras.Input(shape=(T,))
dec_inp = tf.keras.Input(shape=(T,))

enc = tf.keras.layers.Embedding(Vsrc, E, mask_zero=True)(enc_in)
enc = tf.keras.layers.Dropout(0.2)(enc)
enc = tf.keras.layers.Bidirectional(
    tf.keras.layers.LSTM(H, return_sequences=True, return_state=True,
                         dropout=0.2, recurrent_dropout=0.2),
    merge_mode='concat')(enc)
enc_outputs = tf.keras.layers.Concatenate()([enc[0], enc[1]])
enc_outputs = tf.keras.layers.Dense(H, activation='tanh')(enc_outputs)
state_h = tf.keras.layers.Dense(H, activation='tanh')(
    tf.keras.layers.Concatenate()([enc[2], enc[4]]))
state_c = tf.keras.layers.Dense(H, activation='tanh')(
    tf.keras.layers.Concatenate()([enc[3], enc[5]]))

dec = tf.keras.layers.Embedding(Vtgt, E, mask_zero=True)(dec_inp)
dec = tf.keras.layers.Dropout(0.2)(dec)
dec, _, _ = tf.keras.layers.LSTM(H, return_sequences=True, return_state=True,
                                 dropout=0.2, recurrent_dropout=0.2)(dec, initial_state=[state_h, state_c])

# Luong-style attention (projected)
q = tf.keras.layers.Dense(H)(dec)
k = tf.keras.layers.Dense(H)(enc_outputs)
att_scores = tf.keras.layers.Dot(axes=[2,2])([q, k])
att_w = tf.keras.layers.Softmax(axis=-1)(att_scores)
att_out = tf.keras.layers.Dot(axes=[2,1])([att_w, enc_outputs])
dec = tf.keras.layers.Dense(H, activation='tanh')(tf.keras.layers.Concatenate()([dec, att_out]))

out = tf.keras.layers.Dense(Vtgt, activation='softmax')(dec)
model = tf.keras.Model([enc_in, dec_inp], out)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy')

# warmup + timing a few steps
model.train_on_batch([X, dec_in], dec_out)  # warmup JIT/graph
N = 50  # measure over 50 steps
start = time.time()
for _ in range(N):
    model.train_on_batch([X, dec_in], dec_out)
elapsed = time.time() - start
ms_per_step = 1000 * elapsed / N
print(f"{ms_per_step:.1f} ms/step  ->  epoch time ≈ {ms_per_step*202/1000:.1f} s")
print(f"2-epoch training time ≈ {(ms_per_step*404)/1000:.1f} s")

2025-09-30 15:15:36.904021: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4 Pro
2025-09-30 15:15:36.904038: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 48.00 GB
2025-09-30 15:15:36.904042: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 18.72 GB
2025-09-30 15:15:36.904056: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-09-30 15:15:36.904065: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


ValueError: A `Concatenate` layer requires inputs with matching shapes except for the concatenation axis. Received: input_shape=[(None, 50, 128), (None, 64)]

In [ ]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer

import nltk
nltk.download('punkt_tab')

import spacy

In [ ]:
df_fr = pd.read_csv('data/small_vocab_fr.txt', sep='\t', names=['text'])
df_fr.head()

In [ ]:
df_en = pd.read_csv('data/small_vocab_en.txt', sep='\t', names=['text'])
df_en.head()

In [ ]:
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def preprocess_text(text, remove_stopwords=False, language='french', remove_punctuation=False):
    """
    Preprocess text by cleaning and tokenizing
    """
    # Basic text cleaning
    text = text.strip() # .lower()
    def fix_punctuation_spacing(text):
        # Apostrophe: no spaces around it
        if text.find("'") != -1:
            text = text.replace(" '", "'").replace("' ", "'")

        # Hyphen in compound words: no spaces around it
        # Em dash (—) or en dash (–): space before and after for sentence breaks
        if text.find("-") != -1:
            # First handle spaced dashes (likely sentence breaks)
            text = text.replace(" - ", " — ")  # Convert to em dash
            text = text.replace(" -", " —").replace("- ", "— ")
            
            # Replace em dashes back to spaced format
            text = text.replace("—", " — ")
            
            # Clean up multiple spaces around em dashes
            text = re.sub(r'\s*—\s*', ' — ', text)
        
        # Comma: no space before, one space after
        if text.find(",") != -1:
            text = text.replace(" ,", ",")
            # Add space after comma if not already there
            text = re.sub(r',(?!\s)', ', ', text)
            # Fix multiple spaces after comma
            text = text.replace(",  ", ", ")

        # Period: no space before, one space after (except end of text)
        if text.find(".") != -1:
            text = text.replace(" .", ".")
            # Add space after period if not already there and not at end
            text = re.sub(r'\.(?!\s|$)', '. ', text)
            # Fix multiple spaces after period
            text = text.replace(".  ", ". ")
        
        # Semicolon: no space before, one space after
        if text.find(";") != -1:
            text = text.replace(" ;", ";")
            text = re.sub(r';(?!\s)', '; ', text)
            text = text.replace(";  ", "; ")
        
        # Colon: no space before, one space after
        if text.find(":") != -1:
            text = text.replace(" :", ":")
            text = re.sub(r':(?!\s)', ': ', text)
            text = text.replace(":  ", ": ")
        
        # Question mark: no space before, one space after
        if text.find("?") != -1:
            text = text.replace(" ?", "?")
            text = re.sub(r'\?(?!\s|$)', '? ', text)
            text = text.replace("?  ", "? ")
        
        # Exclamation mark: no space before, one space after
        if text.find("!") != -1:
            text = text.replace(" !", "!")
            text = re.sub(r'!(?!\s|$)', '! ', text)
            text = text.replace("!  ", "! ")
        
        # Opening parenthesis: one space before (if not at start), no space after
        if text.find("(") != -1:
            text = re.sub(r'(?<!\s)(?<!^)\(', ' (', text)  # Add space before if not already there
            text = text.replace("( ", "(")  # Remove space after
            text = text.replace("  (", " (")  # Fix double spaces
        
        # Closing parenthesis: no space before, one space after (if not at end)
        if text.find(")") != -1:
            text = text.replace(" )", ")")
            text = re.sub(r'\)(?!\s|$|[.,;:!?])', ') ', text)  # Add space after unless at end or before punctuation
            text = text.replace(")  ", ") ")
        
        # Clean up any multiple spaces
        text = re.sub(r'\s+', ' ', text)
        
        return text.strip()

    text = fix_punctuation_spacing(text)

    # Remove digits
    # cleaned_text = ''.join(char for char in text if not char.isdigit())
    
    # Remove punctuation
    # if remove_punctuation:
    #     for punctuation in string.punctuation:
    #         cleaned_text = cleaned_text.replace(punctuation, ' ')

    # Tokenize
    word_tokens = word_tokenize(text, language=language)
    
    # if asked to remove stopwords
    if remove_stopwords:
        print("Removing stopwords")
        # Remove stop words
        if language == 'french':
            stop_words = set(stopwords.words('french'))
        else:
            stop_words = set(stopwords.words('english'))
    
        tokens_cleaned = [w for w in word_tokens if w not in stop_words and len(w) > 0]
    
        return tokens_cleaned
    
    # # Load relevant language model
    # if language == 'french':
        
    #     nlp = spacy.load('fr_core_news_sm')
    # else:
    #     nlp = spacy.load('en_core_web_sm')

    # def process_text(text):
    #     # this is processing part.
    #     doc = nlp(text)

    #     # Filtering step
    #     filtered_tokens = [token.text for token in doc if not token.is_stop]

    #     print("Filtered Tokens:", filtered_tokens)
    word_tokens = [wt for wt in word_tokens if len(wt) > 0]
    # print(word_tokens)

    return word_tokens

# Apply preprocessing to French data
df_fr['tokens'] = df_fr['text'].apply(lambda x: preprocess_text(x, remove_stopwords= False, language='french'))

# Instantiating the TfidfVectorizer
tf_idf_vectorizer_fr = TfidfVectorizer()
# Training it on the texts
weighted_df_fr = pd.DataFrame(tf_idf_vectorizer_fr.fit_transform(df_fr['text']).toarray(),
                    columns = tf_idf_vectorizer_fr.get_feature_names_out())

# Apply preprocessing to English data
df_en['tokens'] = df_en['text'].apply(lambda x: preprocess_text(x, remove_stopwords= False, language='english'))

# Instantiating the TfidfVectorizer
tf_idf_vectorizer_en = TfidfVectorizer()

# Training it on the texts
weighted_df_en = pd.DataFrame(tf_idf_vectorizer_en.fit_transform(df_en['text']).toarray(),
                    columns = tf_idf_vectorizer_en.get_feature_names_out())

print("French preprocessing complete")
print("English preprocessing complete")

# Rename columns to be specific to each language
df_fr_renamed = df_fr.rename(columns={'text': 'text_fr', 'tokens': 'tokens_fr'})
df_en_renamed = df_en.rename(columns={'text': 'text_en', 'tokens': 'tokens_en'})

# Combine side by side
df_fr_en = pd.concat([df_fr_renamed, df_en_renamed], axis=1)

import csv
with open('data/cleaned_texts.csv', 'w', newline='') as csvfile:
    df_fr_en.to_csv(csvfile, index=False)

In [ ]:
print("\nFrench:", weighted_df_fr.shape[1], "\nEnglish:", weighted_df_en.shape[1])

In [ ]:
# Combine side by side

df_fr_en

In [ ]:
df_fr_en.text_fr[137855]

In [ ]:
df_fr_en.tokens_fr[137855]

In [ ]:
weighted_df_en

In [ ]:
df_fr_en['text_fr'][23]

In [ ]:
df_fr_en['text_fr']
df_fr_en['text_en']

In [ ]:
df_fr_en["tokens_fr"]

In [ ]:
from LSTM_translator import train_translator_from_tokens, load_translator_for_inference, test_translation

In [ ]:

import tensorflow as tf

In [ ]:
# Check current setup
print("TensorFlow version:", tf.__version__)
print("Built with MPS:", tf.config.list_physical_devices('GPU'))

In [ ]:
import torch
if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    x = torch.ones(1, device=mps_device)
    print (x)
else:
    print ("MPS device not found.")

# Training

# LSTM

In [ ]:
# # Must be arm64 (not Rosetta)
# uname -m
# # Expect: arm64

# # Python must also be arm64:
# python -c "import platform, sys; print(platform.machine(), sys.version)"
# # Expect: arm64 and Python 3.11 or 3.12

In [ ]:
import tensorflow as tf

# Try mirrored across all GPUs; if none, single device
try:
    strategy = tf.distribute.MirroredStrategy()  # uses all visible GPUs
    print("MirroredStrategy devices:", strategy.num_replicas_in_sync)
except Exception:
    strategy = tf.distribute.OneDeviceStrategy(device="/CPU:0")
    print("Using OneDeviceStrategy (CPU).")

with strategy.scope():
    # If your function builds/compiles a Keras model internally, this scopes it for multi-replica.
    translator, history = train_translator_from_tokens(df_fr_en, validation_size=1000)

translator.save_model('lstm_translator_distributed')

In [ ]:
# import tensorflow as tf

# print("TF GPUs:", tf.config.list_physical_devices('GPU'))
# if tf.config.list_physical_devices('GPU'):
#     # Mixed precision is great on Apple GPU
#     from tensorflow.keras import mixed_precision
#     mixed_precision.set_global_policy('mixed_float16')

#     # Optional: XLA on GPU too (can help)
#     tf.config.optimizer.set_jit(True)

#     with tf.device('/GPU:0'):
#         translator, history = train_translator_from_tokens(df_fr_en, validation_size=1000)
#     translator.save_model('lstm_translator_metal_fp16')
# else:
#     print("No TF Metal GPU found; falling back to CPU.")

In [ ]:
# import os, tensorflow as tf

# # Threading: use all cores (tune if oversubscribed)
# tf.config.threading.set_intra_op_parallelism_threads(0)  # 0 lets TF choose
# tf.config.threading.set_inter_op_parallelism_threads(0)

# # Enable XLA JIT for graphs (big win for RNNs/LSTMs/Transformers)
# tf.config.optimizer.set_jit(True)  # or tf.config.experimental.enable_mlir_graph_optimization()

# # Optional: ensure deterministic (can hurt perf)
# # os.environ["TF_DETERMINISTIC_OPS"] = "0"

# # If your train_translator_from_tokens accepts datasets, use tf.data optimizations.
# # Otherwise, just leave it — XLA+threads still helps.
# tf.random.set_seed(42)
# with tf.device('/CPU:0'):
#     translator, history = train_translator_from_tokens(
#         df_fr_en,
#         validation_size=1000,
#         # if you expose these inside the function, use:
#         # batch_size=..., shuffle_buffer=..., num_parallel_calls=tf.data.AUTOTUNE, prefetch=tf.data.AUTOTUNE,
#     )

# translator.save_model('lstm_translator_cpu_xla')

In [ ]:
# # Your data structure: df with columns ['tokens_fr', 'tokens_en']
# # Example: df.iloc[0]['tokens_fr'] = ['new', 'jersey', 'est', 'parfois', 'calme', ...]

# tf.random.set_seed(42)

# # Train the model
# # with tf.device('/GPU:0'):
# with tf.device('/CPU:0'):
#     translator, history = train_translator_from_tokens(df_fr_en, validation_size=1000)

# # Save for later use
# translator.save_model('lstm_translator')

# # Test some translations
# test_translation(translator, df_fr_en, n_examples=5)


In [ ]:
translator = load_translator_for_inference('lstm_translator')

# # Method 1: Preprocess then translate tokens manually
# french_tokens = translator.preprocess_french_phrase("Bonjour, comment allez-vous ?")
# english_tokens = translator.translate_tokens(french_tokens)
# print(english_tokens)

# Method 2: Translate entire sentence directly
english_tokens = translator.translate_sentence("il aime la mangue ?")
print(english_tokens)

In [ ]:
# After kernel restart - Fresh test
from LSTM_translator import load_translator_for_inference

translator_loaded = load_translator_for_inference('lstm_translator')
test_tokens = ['new', 'jersey', 'est', 'parfois', 'calme']
result = translator_loaded.translate_tokens(test_tokens)
print(result)

In [ ]:
# Test cell - Loading saved model and attempting prediction
import tensorflow as tf
from LSTM_translator import load_translator_for_inference
import numpy as np
# Load the saved model 
translator_loaded = load_translator_for_inference('lstm_translator')
# Test with a simple French sentence
test_tokens = ['new', 'jersey', 'est', 'parfois', 'calme']
print(f"Testing translation of: {test_tokens}")
# Try to translate (this should fail with same error)
try:
    result = translator_loaded.translate_tokens(test_tokens)
    print(f"Translation: {result}")
except AttributeError as e:
    print(f"Expected error: {e}")
    print("Confirming the issue exists with loaded models too")


In [ ]:

translator = load_translator_for_inference('lstm_translator')

# Translate new French tokens but change to words for vocabulary
french_tokens = ['bonjour', 'comment', 'allez', 'vous']
english_tokens = translator.translate_tokens(french_tokens)
print(english_tokens)  # ['hello', 'how', 'are', 'you']
french_tokens = ["C'est meilleur la banane ?"]
english_tokens = translator.translate_tokens(french_tokens)
print(english_tokens)  # ['Is the banana better?']

In [ ]:
# Method 2: Translate entire sentence directly

print(translator.translate_sentence("c'est bien la banane ?"))
print(translator.translate_sentence("parfois calme parfois neigeux"))

print(translator.translate_sentence("new jersey est parfois calme pendant l' automne , et il est neigeux en avril ."))
print(translator.translate_sentence("le pamplemousse est votre fruit le plus aimé , mais le raisin est leur plus aimé ."))

In [ ]:
# # Recover the exact validation set used during training
# from sklearn.model_selection import train_test_split
# # Recreate the same split (using the same parameters as during training)
# df_temp, df_val = train_test_split(df_fr_en, test_size=1000, random_state=42)
# print(f"Recovered validation set: {len(df_val)} samples")
# # Test your model on the validation set
# def calculate_validation_accuracy(translator, df_val, n_samples=100):
#     """Calculate translation accuracy on validation set"""
#     correct = 0
#     total = min(n_samples, len(df_val))
#     for i in range(total):
#         fr_tokens = df_val.iloc[i]['tokens_fr']
#         true_en_tokens = df_val.iloc[i]['tokens_en']
#         try:
#             predicted_tokens = translator.translate_tokens(fr_tokens)
#             # Simple accuracy: exact match
#             if predicted_tokens == true_en_tokens:
#                 correct += 1
#             if i % 20 == 0:
#                 print(f"Sample {i+1}:")
#                 print(f"  French: {' '.join(fr_tokens)}")
#                 print(f"  True: {' '.join(true_en_tokens)}")
#                 print(f"  Predicted: {' '.join(predicted_tokens)}")
#                 print(f"  Match: {'✓' if predicted_tokens == true_en_tokens else '✗'}")
#         except Exception as e:
#             print(f"Translation failed for sample {i}: {e}")
#     accuracy = correct / total
#     print(f"\nValidation Accuracy: {accuracy:.4f} ({correct}/{total})")
#     return accuracy
# # Use it
# accuracy = calculate_validation_accuracy(translator, df_val, n_samples=1000)


Validation Accuracy: 0.7500 (750/1000)

Sample 81:
  French: la france est belle au mois de novembre , mais il est généralement doux en été .
  True: france is nice during november , but it is usually mild in summer .
  Predicted: france is beautiful during november , but it is usually mild in summer .
...
  Predicted: the strawberry is her most loved fruit , but the grape is my most loved .
  Match: ✗

Validation Accuracy: 0.7500 (750/1000)

In [ ]:
import nltk

hypothesis = ['It', 'is', 'a', 'cat', 'at', 'room']
reference = ['It', 'is', 'a', 'cat', 'inside', 'the', 'room']
#there may be several references
BLEUscore = nltk.translate.bleu_score.sentence_bleu([reference], hypothesis)
print(BLEUscore)

In [ ]:
    from sklearn.model_selection import train_test_split
    import nltk
    from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction
    from rouge_score import rouge_scorer


    # --- Split exactly as before ---
    df_temp, df_val = train_test_split(df_fr_en, test_size=1000, random_state=42)
    print(f"Recovered validation set: {len(df_val)} samples")

    # --- Metric helpers ---

    # BLEU smoothing is important for short sentences to avoid 0s from missing higher-order n-grams.
    _bleu_smoother = SmoothingFunction().method3

    # ROUGE scorer: we’ll report F1 for each variant (more stable for translation than recall-only)
    _rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    def _compute_bleu_per_sentence(reference_tokens, hypothesis_tokens):
        """
        reference_tokens: list[str]
        hypothesis_tokens: list[str]
        Returns sentence BLEU (0..1)
        """
        return sentence_bleu(
            [reference_tokens],                   # list of references; you can add more if you have them
            hypothesis_tokens,
            smoothing_function=_bleu_smoother
        )

    def _compute_rouge_f1(reference_tokens, hypothesis_tokens):
        """
        Compute ROUGE-1/2/L F1 between lists of tokens.
        Returns dict with keys: rouge1, rouge2, rougeL (F1 each).
        """
        # ROUGE expects strings; join tokens with spaces
        ref = " ".join(reference_tokens)
        hyp = " ".join(hypothesis_tokens)
        s = _rouge.score(ref, hyp)
        return {
            "rouge1": s["rouge1"].fmeasure,
            "rouge2": s["rouge2"].fmeasure,
            "rougeL": s["rougeL"].fmeasure,
        }

    def calculate_validation_metrics(translator, df_val, n_samples=1000, log_every=20):
        """
        Evaluate on the validation set with:
        - Exact-match accuracy
        - Mean sentence BLEU
        - Corpus BLEU
        - Mean ROUGE-1/2/L (F1)

        Returns a dict of aggregate metrics.
        """
        correct = 0
        total = min(n_samples, len(df_val))
        bleu_scores = []
        rouge1_scores = []
        rouge2_scores = []
        rougeL_scores = []

        # For corpus BLEU we need aggregated references and hypotheses
        corpus_references = []  # list of list-of-references (each reference is a list of tokens)
        corpus_hypotheses = []  # list of hypothesis token lists

        for i in range(total):
            fr_tokens = df_val.iloc[i]['tokens_fr']
            true_en_tokens = df_val.iloc[i]['tokens_en']
            try:
                predicted_tokens = translator.translate_tokens(fr_tokens)

                # --- Exact match accuracy ---
                if predicted_tokens == true_en_tokens:
                    correct += 1

                # --- Sentence BLEU ---
                bleu = _compute_bleu_per_sentence(true_en_tokens, predicted_tokens)
                bleu_scores.append(bleu)

                # --- ROUGE (F1) ---
                r = _compute_rouge_f1(true_en_tokens, predicted_tokens)
                rouge1_scores.append(r["rouge1"])
                rouge2_scores.append(r["rouge2"])
                rougeL_scores.append(r["rougeL"])

                # --- Accumulate for corpus BLEU ---
                corpus_references.append([true_en_tokens])  # wrap in list: potentially multiple refs
                corpus_hypotheses.append(predicted_tokens)

                # --- Optional logging ---
                if i % log_every == 0:
                    print(f"Sample {i+1}:")
                    print(f"  French:    {' '.join(fr_tokens)}")
                    print(f"  True:      {' '.join(true_en_tokens)}")
                    print(f"  Predicted: {' '.join(predicted_tokens)}")
                    print(f"  Match:     {'✓' if predicted_tokens == true_en_tokens else '✗'}")
                    print(f"  BLEU:      {bleu:.4f} | ROUGE-1/2/L (F1): "
                        f"{r['rouge1']:.4f}/{r['rouge2']:.4f}/{r['rougeL']:.4f}")

            except Exception as e:
                print(f"Translation failed for sample {i}: {e}")

        # --- Aggregates ---
        accuracy = correct / total if total > 0 else 0.0
        mean_sentence_bleu = sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0.0
        # corpus BLEU (usually reported for MT), also with smoothing to handle short texts robustly
        corpus_bleu_score = corpus_bleu(corpus_references, corpus_hypotheses, smoothing_function=_bleu_smoother) if corpus_hypotheses else 0.0
        mean_rouge1 = sum(rouge1_scores) / len(rouge1_scores) if rouge1_scores else 0.0
        mean_rouge2 = sum(rouge2_scores) / len(rouge2_scores) if rouge2_scores else 0.0
        mean_rougeL = sum(rougeL_scores) / len(rougeL_scores) if rougeL_scores else 0.0

        # --- Report ---
        print("\n=== Validation Summary ===")
        print(f"Accuracy (exact match): {accuracy:.4f} ({correct}/{total})")
        print(f"Mean sentence BLEU:     {mean_sentence_bleu:.4f}")
        print(f"Corpus BLEU:            {corpus_bleu_score:.4f}")
        print(f"ROUGE-1/2/L (F1 mean):  {mean_rouge1:.4f} / {mean_rouge2:.4f} / {mean_rougeL:.4f}")

        return {
            "accuracy": accuracy,
            "mean_sentence_bleu": mean_sentence_bleu,
            "corpus_bleu": corpus_bleu_score,
            # "mean_rouge1_f1": mean_rouge1,
            # "mean_rouge2_f1": mean_rouge2,
            "mean_rougeL_f1": mean_rougeL,
        }

    # --- Example usage (replaces your previous call) ---
    metrics = calculate_validation_metrics(translator, df_val, n_samples=1000)

## GRU

In [ ]:
from GRU_translator import GRUTranslator, train_translator_from_tokens

# Train GRU model (uses same config as LSTM)
gru_translator, history = train_translator_from_tokens(df_fr_en, validation_size=1000)

# Save and load
gru_translator.save_model('gru_translator')
loaded_gru = GRUTranslator.load_model('gru_translator')

# Translate
result = gru_translator.translate_tokens(['hello', 'world'])

In [ ]:
df_fr_en['text_fr'][14230]

In [ ]:
df_fr_en["text_en"]

In [ ]:
import nltk

hypothesis = ['It', 'is', 'a', 'cat', 'at', 'room']
reference = ['It', 'is', 'a', 'cat', 'inside', 'the', 'room']
#there may be several references
BLEUscore = nltk.translate.bleu_score.sentence_bleu([reference], hypothesis)
print(BLEUscore)

In [ ]:
import pandas as pd
from collections import Counter

# Assuming your data is in df_fr_en["text_en"]
all_text = " ".join(df_fr_en["text_en"].astype(str))
words = all_text.split(" ")
word_counts = Counter(words)

# Display results
for word, count in word_counts.most_common():
    print(f"{word}: {count}")

In [ ]:
import pandas as pd
from collections import Counter

# Assuming your data is in df_fr_en["text_en"]
all_text = " ".join(df_fr_en["text_fr"].astype(str))
words = all_text.split(" ")
word_counts = Counter(words)

# Display results
for word, count in word_counts.most_common():
    print(f"{word}: {count}")

In [ ]:
df_fr_en["text_en"]

## RNN

In [ ]:
from RNN_translator import RNNTranslator, grid_search_rnn

# Define parameter grid
param_grid = {
    'embedding_dim': [64, 128],
    'hidden_units': [512],
    'dropout_rate': [0.1],
    'learning_rate': [0.001, 0.0001]
}

# Run grid search
best_configs = grid_search_rnn(df_fr_en, param_grid, epochs=30, patience=5, n_best=3)
# Train with best config
best_params = best_configs[0]['params']
rnn = RNNTranslator(**best_params)

configuration 1 completed:
   Test Accuracy: 0.3525
   Test Loss: 1.9067
   Total Parameters: 731,660


## Faster LSTM ATTEMPT